# Subset Selection Notebook 1: Data Preparation & Configuration
This notebook focusses on setting up the environment, configuring the subset selection process, and preparing the input data

## Introduction and Setup

In [1]:
# Install the necessary libraries
%pip install datasets jinja2 tqdm h5py numpy torch transformers submodlib-py==0.0.3 seaborn

Note: you may need to restart the kernel to use updated packages.


## Imports and Logging

In [2]:
# Standard Imports
from dataclasses import dataclass, field
from typing import Any, Dict, List, TypedDict, TypeVar, Union
import logging
import os
import re

# Third Party Imports
from datasets import load_dataset, concatenate_datasets
from jinja2 import BaseLoader, Environment
from tqdm import tqdm
import torch
import numpy as np
import warnings

# Configure logging and warnings
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore", category=UserWarning)

/Users/roburishabh/Github/odh-data-processing/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Core Utility Functions

Defines three essential utility functions used throughout the pipeline:
1. **`get_default_num_gpus(testing_mode)`**
   - Detects available GPUs for parallel processing
   - Falls back to CPU in testing mode
   - Returns: Number of GPUs to use

2. **`retry_on_exception(func)`**
   - Decorator for automatic retry on common errors
   - Handles GPU OOM, runtime errors, value errors
   - Cleans up GPU memory between retries

3. **`get_supported_encoders()`**
   - Dynamically discovers available encoder types
   - Searches for encoder files in the encoders directory
   - Returns: List of encoder names (e.g., ['arctic'])

In [ ]:
import glob
from functools import wraps
import gc
import time

def get_default_num_gpus(testing_mode: bool = False) -> int:
    """
    Get the default number of GPUs based on available CUDA devices.
    
    Args:
        testing_mode (bool): If True, allows CPU usage with warnings. For testing only.
    """
    if not torch.cuda.is_available():
        if testing_mode:
            logger.warning(
                "No CUDA devices detected. Running in testing mode with CPU. "
                "Production use requires GPU acceleration."
            )
            return 1
        raise RuntimeError(
            "No CUDA devices detected. This functionality requires at least one GPU."
        )
    return torch.cuda.device_count()


def retry_on_exception(func):
    """
    Decorator to retry a function upon exception up to a maximum number of retries.
    """
    @wraps(func)
    def wrapper(self, *args, **kwargs):
        last_exception = None
        for attempt in range(self.config.system.max_retries):
            try:
                return func(self, *args, **kwargs)
            except torch.cuda.OutOfMemoryError as e:
                last_exception = e
                logger.error(f"GPU out of memory on attempt {attempt + 1}: {str(e)}")
            except RuntimeError as e:
                last_exception = e
                logger.error(f"PyTorch runtime error on attempt {attempt + 1}: {str(e)}")
            except ValueError as e:
                last_exception = e
                logger.error(f"Value error on attempt {attempt + 1}: {str(e)}")
            except TypeError as e:
                last_exception = e
                logger.error(f"Type error on attempt {attempt + 1}: {str(e)}")
            except IndexError as e:
                last_exception = e
                logger.error(f"Index error on attempt {attempt + 1}: {str(e)}")

            if attempt < self.config.system.max_retries - 1:
                logger.info(f"Retrying in {self.config.system.retry_delay} seconds...")
                time.sleep(self.config.system.retry_delay)
                gc.collect()
                torch.cuda.empty_cache()

        raise last_exception

    return wrapper


def get_supported_encoders():
    """Get list of supported encoder types from the encoders directory."""
    try:
        # Try to find encoders directory relative to notebook
        import sys
        from pathlib import Path
        
        # Get the notebook's directory
        notebook_dir = Path.cwd()
        
        # Look for encoders directory in parent directory structure
        possible_paths = [
            notebook_dir.parent / "encoders",  # ../encoders
            notebook_dir / "encoders",          # ./encoders
            Path(__file__).parent / "encoders" if '__file__' in globals() else None
        ]
        
        for encoders_dir in possible_paths:
            if encoders_dir and encoders_dir.exists():
                encoder_files = list(encoders_dir.glob("*_encoder.py"))
                if encoder_files:
                    return [
                        f.stem.replace("_encoder", "")
                        for f in encoder_files
                        if not f.name.startswith("__")
                    ]
        
        # Fallback: return known encoders
        logger.warning("Could not dynamically discover encoders. Using fallback list.")
        return ["arctic"]
        
    except Exception as e:
        logger.warning(f"Error discovering encoders: {e}. Using fallback list.")
        return ["arctic"]


print("✅ Utility functions defined!")

✅ Utility functions defined!


## Configuration Classes

### BasicConfig Class
It defines the basic processing pramaters. It centralizes basic configuration with validation and helpful metadata.
- output_dir: Where results will be saved
- batch_size: Number of samples processed in each batch (100K for efficiency)
- num_folds: Number of folds for cross-validation in subset selection
- combine_files: Whether to combine multiple input files
- epsilon: Optimization parameter for submodular selection (160.0 for large datasets)

In [4]:
@dataclass
class BasicConfig:
    """Basic configuration parameters"""
    output_dir: str = "notebooks/data/outputs" # change this to your desired output directory
    batch_size: int = 100000
    num_folds: int = 50
    combine_files: bool = False
    epsilon: float = field(
        default=160.0,
        metadata={
            "advanced": True,
            "help": "Epsilon parameter for the LazierThanLazyGreedy optimizer in facility location maximization. "
            "Default of 160.0 is optimized for datasets >100k samples. "
            "For smaller datasets, consider using much smaller values (starting from 0.1).",
        },
    )

    def __post_init__(self):
        """Validate configuration after initialization"""
        if not 0 < self.epsilon <= 160:
            raise ValueError("epsilon must be between 0 and 160")

    def validate_epsilon_for_dataset_size(self, dataset_size: int)->None:
        """
        Validate epsilon parameter based on dataset size and provide appropriate warnings.

        Args:
            dataset_size (int): Size of the dataset being processed
        """
        if dataset_size < 100000:
            logger.warning(
                "Subset selection is highly recommended to be used only with dataset sizes over 100k samples. "
                f"Your dataset has {dataset_size:,} samples."
            )
            if self.epsilon > 1.0:
                logger.warning(
                    f"Current epsilon value ({self.epsilon}) may be too high for a dataset of this size. "
                    "For smaller datasets, consider using much smaller values (starting from 0.1) "
                    "to ensure proper subset selection."
                )

### EncoderConfig Class
It configures the embedding encoder, it basically separates encoder configuration from other parameters for modularity.
- instruction: Prompt for the encoder to understand the task
- encoder_type: Type of encoder to use (arctic, bge, etc.)
- encoder_model: Specific model name
- testing_mode: Enables development/testing features

In [5]:
@dataclass
class EncoderConfig:
    """Encoder-specific configuration parameters."""
    instruction: str = field(
        default="Generate embeddings that capture the core meaning of user-assistant conversations, ensuring the embeddings can be clustered based on semantic similarity for subset selection.",
        metadata={"advanced": True},
    )
    encoder_type: str = field(default="arctic", metadata={"advanced": True})
    encoder_model: str = field(
        default="Snowflake/snowflake-arctic-embed-l-v2.0", metadata={"advanced": True}
    )
    testing_mode: bool = False

### TemplateConfig class
It manages text formatting templates, so that we get flexible text formatting foir different data structures.
- template_name: Active template to use
- templates: Dictionary of available templates (conversation, qa, default)
- Uses Jinja2 syntax for flexible text formatting

In [6]:
@dataclass
class TemplateConfig:
    """Template-related configuration parameters."""
    template_name: str = field(default="conversation", metadata={"advanced": True})
    templates: Dict[str, str] = field(
        default_factory=lambda: {
            "default": "{{ text }}",
            "conversation": "{% for msg in messages if msg.role != 'system' %}{{ msg.role }}: {{ msg.content }}\n{% endfor %}",
            "qa": "Question: {{ question }}\nAnswer: {{ answer }}",
        },
        metadata={"advanced": True},
    )

### SystemConfig class
It manages system-level configuration for handling system resources and error recovery.
- num_gpus: Auto-detects available GPUs
- seed: Random seed for reproducibility
- max_retries: Number of retry attempts for failed operations
- retry_delay: Delay between retries
- testing_mode: Enables testing features

In [7]:
@dataclass
class SystemConfig:
    """System-related configuration parameters."""
    num_gpus: int = field(init=False)  # Don't initialize in __init__
    seed: int = field(default=42, metadata={"advanced": True})
    max_retries: int = field(default=3, metadata={"advanced": True})
    retry_delay: int = field(default=30, metadata={"advanced": True})
    testing_mode: bool = field(default=False, metadata={"advanced": True})

    def __post_init__(self):
        """Initialize num_gpus after other fields are set."""
        # Use the standalone utility function instead of duplicating logic
        self.num_gpus = get_default_num_gpus(testing_mode=self.testing_mode)

### ProcessingConfig class
Its the main configuration class that combines all other configuration, it provides a single point of configuration with validation. 
- input_files: List of input files to process
- subset_sizes: List of subset sizes - integers for absolute counts or floats for percentages
- Combines all configuration groups into one object
- Validates configuration parameters

Configuration Groups:
- basic: Basic processing parameters
- encoder: Encoder-specific parameters
- template: Template-related parameters
- system: System-related parameters

In [8]:
@dataclass
class ProcessingConfig:
    """
    Configuration for subset selection with basic and advanced parameters.
    
    """
    # Required parameters
    input_files: List[str]
    subset_sizes: List[Union[int, float]]

    # Configuration groups
    basic: BasicConfig = field(default_factory=BasicConfig)
    encoder: EncoderConfig = field(default_factory=EncoderConfig)
    template: TemplateConfig = field(default_factory=TemplateConfig)
    system: SystemConfig = field(default_factory=SystemConfig)

    def __post_init__(self):
        """Validate configuration after initialization."""
        if not isinstance(self.subset_sizes, list):
            raise ValueError("subset_sizes must be a list")

        for size in self.subset_sizes:
            if not isinstance(size, (int, float)):
                raise ValueError("subset_sizes must contain only integers or floats")
            if isinstance(size, float) and not 0 < size <= 100:
                raise ValueError(
                    "Percentage values in subset_sizes must be between 0 and 100"
                )
            if isinstance(size, int) and size <= 0:
                raise ValueError("Absolute values in subset_sizes must be positive")

## Data Processor Class
It is the main processing class with utility functions, that centralizes all data processing logic with proper error handling.
- Handles data loading and formatting
- Manages template application
- Provides utility functions for subset calculations
- Maintains configuration and state

In [9]:
class DataProcessor:
    """
    Enhanced data processor with support for combined files and multiple selection methods.
    
    This class handles the complete pipeline:
    - Data loading and formatting
    - Embedding generation (to be implemented in Notebook 2)
    - Subset selection (to be implemented in Notebook 3)
    - Result export
    """

    def __init__(self, config: ProcessingConfig):
        """
        Initializes the DataProcessor with the given configuration.

        Args:
            config (ProcessingConfig): The processing configuration.
        """
        self.config = config
        self.env = Environment(loader=BaseLoader())
        self.templates = {
            k: self.env.from_string(v) for k, v in config.template.templates.items()
        }
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Set random seeds
        np.random.seed(config.system.seed)
        torch.manual_seed(config.system.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(config.system.seed)

    def format_text(self, example: Dict[str, Any], format_type: str) -> str:
        """
        Formats the text of an example using the specified template.

        Args:
            example (Dict[str, Any]): The data example to format.
            format_type (str): The key of the template to use.

        Returns:
            str: The formatted text.
        """
        template = self.templates.get(format_type)
        if not template:
            raise ValueError(f"Unknown format type: {format_type}")
        return template.render(**example)

    def load_and_combine_datasets(self, input_files: List[str]):
        """
        Load and optionally combine multiple datasets.

        Args:
            input_files (List[str]): List of input file paths.

        Returns:
            Combined dataset or list of individual datasets.
        """
        datasets = []

        for input_file in input_files:
            file_extension = input_file.split(".")[-1]
            if file_extension == "jsonl":
                file_extension = "json"
            dataset = load_dataset(
                file_extension, data_files=input_file, split="train", cache_dir=None
            )
            datasets.append(dataset)

        if self.config.basic.combine_files:
            logger.info("Combining datasets...")
            return concatenate_datasets(datasets)

        if len(datasets) > 1:
            raise ValueError(
                "Multiple datasets provided but combine_files is not enabled"
            )
        return datasets[0]

    def calculate_subset_size(self, total_samples: int, size_spec: Union[int, float]) -> int:
        """
        Calculate the actual subset size based on the specification.
        
        Args:
            total_samples (int): Total number of samples in the dataset.
            size_spec (Union[int, float]): Size specification (percentage if float, absolute if int).

        Returns:
            int: Actual number of samples to select.
        """
        if isinstance(size_spec, float):
            # Handle percentage (0.1 = 10%, 0.05 = 5%)
            if size_spec <= 0 or size_spec > 1:
                raise ValueError(
                    "Percentage values must be between 0(non-inclusive) and 1(inclusive)"
                )
            return max(1, int(size_spec * total_samples))
        # Treat as absolute number
        return min(size_spec, total_samples)

    def get_subset_name(self, size_spec: Union[int, float], actual_size: int) -> str:
        """
        Generate appropriate subset name based on selection method.

        Args:
            size_spec (Union[int, float]): Original size specification.
            actual_size (int): Actual number of samples selected.

        Returns:
            str: Descriptive name for the subset.
        """
        if isinstance(size_spec, float):
            return f"percent_{size_spec:.1f}"
        return f"samples_{actual_size}"

    def get_dataset_name(self, input_file: str) -> str:
        """
        Get a clean dataset name from the input file path.

        Args:
            input_file (str): Input file path

        Returns:
            str: Clean dataset name
        """
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        clean_name = re.sub(r"[^\w\-_]", "_", base_name)
        return clean_name

print("✅ DataProcessor class defined successfully with all utility functions!")

✅ DataProcessor class defined successfully with all utility functions!


## Configuration Setup
### Create configuration for subset selection 
- Sets up input files and subset sizes
- Configures basic processing parameters
- Sets up encoder configuration

In [10]:
config = ProcessingConfig(
    input_files=["data/combined_cut_50x.jsonl"],  # Update with your data path
    subset_sizes=[0.1, 0.05],  # 10% and 5% subsets
    basic=BasicConfig(
        output_dir="output",
        batch_size=100000,
        num_folds=25,
        epsilon=160.0,
        combine_files=False
    ),
    encoder=EncoderConfig(
        encoder_type="arctic",
        encoder_model="Snowflake/snowflake-arctic-embed-l-v2.0",
        testing_mode=True  # Enable for notebook development
    ),
    template=TemplateConfig(
        template_name="conversation",
        templates={
            "default": "{{ text }}",
            "conversation": "{% for msg in messages if msg.role != 'system' %}{{ msg.role }}: {{ msg.content }}\n{% endfor %}",
            "qa": "Question: {{ question }}\nAnswer: {{ answer }}",
        }
    ),
    system=SystemConfig(
        seed=42,
        testing_mode=True,
        max_retries=3,
        retry_delay=30
    )
)

print("===== Configuration created successfully! =====")
print(f"Input files: {config.input_files}")
print(f"Subset sizes: {config.subset_sizes}")
print(f"Output directory: {config.basic.output_dir}")
print(f"Encoder type: {config.encoder.encoder_type}")
print(f"Template name: {config.template.template_name}")
print(f"Number of GPUs: {config.system.num_gpus}")
print(f"Testing mode: {config.system.testing_mode}")

2025-10-10 00:29:02 - WARNING - __main__ - No CUDA devices detected. Running in testing mode with CPU. Production use requires GPU acceleration.


===== Configuration created successfully! =====
Input files: ['data/combined_cut_50x.jsonl']
Subset sizes: [0.1, 0.05]
Output directory: output
Encoder type: arctic
Template name: conversation
Number of GPUs: 1
Testing mode: True


## Data Loading and Validate input data
- Checks if data file exists
- Loads dataset using HuggingFace datasets
- Shows dataset statistics

In [11]:
print("📁 Data Loading")
print("=" * 50)

# Check if data file exists
data_file = config.input_files[0]
if not os.path.exists(data_file):
    print(f"❌ Data file not found: {data_file}")
    print("Please update the data_file path in the configuration above")
    print("Example data file structure:")
    print("""
    [
        {
            "messages": [
                {"role": "user", "content": "Hello, how are you?"},
                {"role": "assistant", "content": "I'm doing well, thank you!"}
            ]
        },
        ...
    ]
    """)
else:
    print(f"✅ Found data file: {data_file}")
    
    # Load the dataset
    try:
        dataset = load_dataset("json", data_files=data_file, split="train", cache_dir=None)
        print(f"✅ Dataset loaded successfully!")
        print(f"📊 Dataset size: {len(dataset):,} samples")
        
        # Show file size
        file_size = os.path.getsize(data_file) / 1024**2  # MB
        print(f"📁 File size: {file_size:.1f} MB")
        
    except Exception as e:
        print(f"❌ Error loading dataset: {e}")
        dataset = None

📁 Data Loading
✅ Found data file: data/combined_cut_50x.jsonl
✅ Dataset loaded successfully!
📊 Dataset size: 1,855 samples
📁 File size: 177.8 MB


## Data Inspection - show data structure
It analyzes loaded data structure and quality
- Displays sample data
- Analyzes column structure
- Counts messages and roles
- validates epsilon for dataset size

In [12]:
if dataset is not None:
    print("🔍 Data Inspection")
    print("=" * 50)
    
    # Show sample data
    print("📋 Sample data:")
    for i in range(min(3, len(dataset))):
        print(f"\nSample {i+1}:")
        sample = dataset[i]
        for key, value in sample.items():
            if isinstance(value, str) and len(value) > 100:
                print(f"  {key}: {value[:100]}...")
            else:
                print(f"  {key}: {value}")
    
    # Analyze data structure
    print(f"\n📊 Data structure analysis:")
    print(f"   Number of samples: {len(dataset):,}")
    
    # Check column names
    if hasattr(dataset, 'column_names'):
        print(f"   Column names: {dataset.column_names}")
    
    # Analyze message structure if it exists
    if 'messages' in dataset.column_names:
        message_lengths = []
        role_counts = {'user': 0, 'assistant': 0, 'system': 0}
        
        for sample in dataset.select(range(min(1000, len(dataset)))):
            if 'messages' in sample:
                message_lengths.append(len(sample['messages']))
                for msg in sample['messages']:
                    if 'role' in msg:
                        role_counts[msg['role']] += 1
        
        print(f"Average messages per conversation: {np.mean(message_lengths):.1f}")
        print(f"Role distribution: {role_counts}")
    
    # Validate epsilon for dataset size
    config.basic.validate_epsilon_for_dataset_size(len(dataset))
    
else:
    print("❌ No dataset loaded. Please fix the data loading issue above.")

2025-10-10 00:29:03 - WARNING - __main__ - Subset selection is highly recommended to be used only with dataset sizes over 100k samples. Your dataset has 1,855 samples.
2025-10-10 00:29:03 - WARNING - __main__ - Current epsilon value (160.0) may be too high for a dataset of this size. For smaller datasets, consider using much smaller values (starting from 0.1) to ensure proper subset selection.


🔍 Data Inspection
📋 Sample data:

Sample 1:
  question: When a reporting entity relies on another reporting entity or an affiliated foreign entity to verify...
  document_outline: The document contains excerpts from FINTRAC regulations designed to combat money laundering and terr...
  raw_document: the identity of a person, you must: 24 as soon as feasible, obtain from the other RE or affiliated f...
  document: ### Extracts from the Passage  

1. > “If you rely on another RE or an affiliated foreign entity to ...
  messages: [{'role': 'user', 'content': "The document contains excerpts from FINTRAC regulations designed to combat money laundering and terrorist financing in Canada\n### Extracts from the Passage  \n\n1. > “If you rely on another RE or an affiliated foreign entity to verify the identity of a person, you must keep a record of: the person’s name; the written agreement or arrangement with the other RE or affiliated foreign entity for the purpose of verifying a person’s identi

## Template System Setup
- Creates DataProcessor instance
- Shows available templates
- Tests template application
- Validates template compatibility

In [13]:
if dataset is not None:
    print("🎨 Template System Setup")
    print("=" * 50)
    
    # Initialize processor
    processor = DataProcessor(config)
    
    print(f"✅ DataProcessor initialized")
    print(f"📝 Available templates: {list(processor.templates.keys())}")
    print(f"🎯 Active template: {config.template.template_name}")
    
    # Show template content
    active_template = config.template.templates[config.template.template_name]
    print(f"\n📋 Active template content:")
    print(f"   {active_template}")
    
    # Test template with sample data
    print(f"\n🧪 Testing template with sample data:")
    try:
        sample_data = dataset[0]
        formatted_text = processor.format_text(sample_data, config.template.template_name)
        print(f"✅ Template applied successfully!")
        print(f"📝 Formatted text preview:")
        print(f"   {formatted_text[:200]}...")
        
    except Exception as e:
        print(f"❌ Template application failed: {e}")
        print("This might be due to mismatched data structure and template")
        
else:
    print("❌ Cannot test templates without loaded dataset.")

🎨 Template System Setup
✅ DataProcessor initialized
📝 Available templates: ['default', 'conversation', 'qa']
🎯 Active template: conversation

📋 Active template content:
   {% for msg in messages if msg.role != 'system' %}{{ msg.role }}: {{ msg.content }}
{% endfor %}

🧪 Testing template with sample data:
✅ Template applied successfully!
📝 Formatted text preview:
   user: The document contains excerpts from FINTRAC regulations designed to combat money laundering and terrorist financing in Canada
### Extracts from the Passage  

1. > “If you rely on another RE or ...
